In [2]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-05 14:19:23 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



In [21]:
helper.obtener_ultima_ingestion('resultados_vspc_clientes.master_customer_data')

2026-01-16 16:40:42 - [INFO] - Buscando fechas para resultados_vspc_clientes.master_customer_data
2026-01-16 16:40:42 - [INFO] - Transcurrido: 11413, Tiempo de Refresco = 1000
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or datab

{'year': 2026, 'month': 1, 'day': 15}

# Introducción

Se evidencia que los registros en la tabla `resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf` las vinculaciones de adquirencia no comparte registros con la tabla `resultados_wompi.wompi_merchants` que contiene las vinculaciones a wompi

In [22]:
# HASTA QUE AÑO MES SE HAN ACTUALIZADO LAS TRXS
periodo_actual_trxs = '202511'
periodo_actual_trxs

'202511'

# Evolución vinculación aceptación comercios [Adquirencia + Wompi]

## Adquirencia

In [23]:
dict_ult_ing_adqu_vinc = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf')
dict_ult_ing_adqu_vinc

2026-01-16 16:40:47 - [INFO] - Buscando fechas para resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2026-01-16 16:40:47 - [INFO] - Finalizo la busqueda, duracion: 00:00.3, resultado: {'year': 2026, 'month': 1, 'day': 7}


{'year': 2026, 'month': 1, 'day': 7}

In [24]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_hist_vinc_adqu PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_hist_vinc_adqu STORED AS PARQUET AS WITH vinculados AS
  (SELECT codigo_unico,
          min(periodo) AS fecha_ym
   FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
   WHERE YEAR <= """ + str(dict_ult_ing_adqu_vinc['year']) + """
     AND MONTH BETWEEN 1 AND 12
     AND DAY BETWEEN 1 AND 31
     AND periodo IS NOT NULL
   GROUP BY 1),
                                                                                       conteo AS
  (SELECT fecha_ym,
          count(*) AS num_vinc_new,
          cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes
   FROM vinculados
   GROUP BY 1)
SELECT fecha_ym,
       num_vinc_new,
       sum(num_vinc_new) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym,
                          'adquirencia' AS producto
FROM conteo
ORDER BY fecha_ym DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_hist_vinc_adqu;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 21/21      DROP ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   04:40:47 PM     00:00.1 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 22/22    CREATE ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   04:40:47 PM     00:01.4 
-------------------------------------------------------------------------------------------------
--------------------

## Wompi

In [4]:
dict_ult_ing_wompi_merch = helper.obtener_ultima_ingestion('resultados_wompi.wompi_merchants')
dict_ult_ing_wompi_merch


2026-02-05 14:46:50 - [INFO] - Buscando fechas para resultados_wompi.wompi_merchants
2026-02-05 14:46:50 - [INFO] - Transcurrido: 1639, Tiempo de Refresco = 1000
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2026-02-05 14:46:51 - [INFO] - Finalizo la busqueda, duracion: 00:00.6, resultado: {'year': 2026, 'month': 2, 'day': 5}


{'year': 2026, 'month': 2, 'day': 5}

In [ ]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_hist_vinc_wompi PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_hist_vinc_wompi STORED AS PARQUET AS
WITH conteo AS (
SELECT CASt(left(cast(creado as string), 6) as int) as fecha_ym, 
       count(*) as num_vinc_new
FROM resultados_wompi.wompi_merchants
WHERE YEAR = """ + str(dict_ult_ing_wompi_merch['year']) + """
  AND MONTH = """ + str(dict_ult_ing_wompi_merch['month']) + """
  AND DAY = """ + str(dict_ult_ing_wompi_merch['day']) + """
        AND modelo = 'Agregador'
        AND activo = 'A'
        AND desembolsos_permitidos = 'Si'
        GROUP BY 1),
        conteo_y_m AS
        (SELECT fecha_ym,
                num_vinc_new,
                cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
                cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes,
                1 AS paracumsum
        FROM conteo)
        SELECT fecha_ym,
        num_vinc_new,
        sum(num_vinc_new) OVER (PARTITION BY YEAR
                                ORDER BY YEAR, mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym,
        sum(num_vinc_new) OVER (PARTITION BY paracumsum
                                ORDER BY fecha_ym) AS num_vinc_new_cumsum,
        'wompi' AS producto
        FROM conteo_y_m
        ORDER BY fecha_ym DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_hist_vinc_wompi;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 24/24      DROP ..._aceptacion_comercios_hist_vinc_wompi   finalizado   04:40:50 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 25/25    CREATE ..._aceptacion_comercios_hist_vinc_wompi   finalizado   04:40:50 PM     00:00.6 
-------------------------------------------------------------------------------------------------
--------------------

# Vinculación aceptación comercios

In [27]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_vinc PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_vinc STORED AS PARQUET AS
WITH junte AS
  (SELECT fecha_ym,
          num_vinc_new,
          num_vinc_new_cumsum_ym,
          producto
   FROM proceso.mdo_aceptacion_comercios_hist_vinc_adqu
   UNION ALL SELECT fecha_ym,
                    num_vinc_new,
                    num_vinc_new_cumsum_ym,
                    producto
   FROM proceso.mdo_aceptacion_comercios_hist_vinc_wompi),
     agregado AS
  (SELECT fecha_ym,
          cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes,
          sum(num_vinc_new) AS num_vinc_new,
          1 AS secuencia2
   FROM junte
   GROUP BY 1,
            2,
            3)
SELECT fecha_ym,
       num_vinc_new,
       sum(num_vinc_new) OVER (PARTITION BY secuencia2 ORDER BY fecha_ym ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_cumsum,
       sum(num_vinc_new) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym
FROM agregado
ORDER BY fecha_ym DESC;
"""
df_outcome = helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_vinc;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 27/27      DROP    proceso.mdo_aceptacion_comercios_vinc   finalizado   04:40:52 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 28/28    CREATE    proceso.mdo_aceptacion_comercios_vinc   finalizado   04:40:52 PM     00:00.3 
-------------------------------------------------------------------------------------------------
--------------------

In [28]:
sql = """
SELECT fecha_ym,
       num_vinc_new,
       num_vinc_cumsum,
       num_vinc_new_cumsum_ym
FROM proceso.mdo_aceptacion_comercios_vinc
ORDER BY fecha_ym DESC
"""
df_outcome = helper.obtener_dataframe(sql)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 30/30 DATAFRAME                                            ejecutando   04:40:53 PM             

2026-01-16 16:40:54 - [INFO] - 88 filas, 4 columnas, 00:00.5 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 30/30 DATAFRAME                                            finalizado   04:40:53 PM     00:00.7 
-------------------------------------------------------------------------------------------------


In [29]:
df_outcome[40:].head(20)

,fecha_ym,num_vinc_new,num_vinc_cumsum,num_vinc_new_cumsum_ym
40,202209.0,5755,166921,46334
41,202208.0,6303,161166,40579
42,202207.0,5110,154863,34276
43,202206.0,5803,149753,29166
44,202205.0,5311,143950,23363
45,202204.0,4498,138639,18052
46,202203.0,5760,134141,13554
47,202202.0,4404,128381,7794
48,202201.0,3390,123977,3390
49,202112.0,4711,120587,68234


# Uso de vinculados nuevos aceptación comercios

In [30]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_num_vinc_new_uso PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_num_vinc_new_uso STORED AS PARQUET AS
WITH uso_adqui AS
  (SELECT periodo,
          count(*) AS num_vinc_new_uso_adqu
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'adqui'
   AND tipo_cliente = 'nuevos'
   GROUP BY 1),
     uso_wompi AS
  (SELECT periodo,
          count(*) AS num_vinc_new_uso_womp
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'wompi'
   AND tipo_cliente = 'nuevos'
   GROUP BY 1)
SELECT a.periodo as fecha_ym,
       a.num_vinc_new_uso_adqu + nvl(b.num_vinc_new_uso_womp, 0) AS num_vinc_new_uso_cumsum_ym
FROM uso_adqui AS a
LEFT JOIN uso_wompi AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_num_vinc_new_uso;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 31/31      DROP ...aceptacion_comercios_num_vinc_new_uso   finalizado   04:40:54 PM     00:00.1 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 32/32    CREATE ...aceptacion_comercios_num_vinc_new_uso   finalizado   04:40:54 PM     00:05.3 
-------------------------------------------------------------------------------------------------
--------------------

# Uso de vinculados viejos aceptación comercios

In [31]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_num_vinc_old_uso PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_num_vinc_old_uso STORED AS PARQUET AS
WITH uso_adqui AS
  (SELECT periodo,
          count(*) AS num_vinc_old_uso_adqu
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'adqui'
   AND tipo_cliente = 'viejos'
   GROUP BY 1),
     uso_wompi AS
  (SELECT periodo,
          count(*) AS num_vinc_old_uso_womp
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'wompi'
   AND tipo_cliente = 'viejos'
   GROUP BY 1)
SELECT a.periodo as fecha_ym,
       a.num_vinc_old_uso_adqu + nvl(b.num_vinc_old_uso_womp, 0) AS num_vinc_old_uso_cumsum_ym
FROM uso_adqui AS a
LEFT JOIN uso_wompi AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_num_vinc_old_uso;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 34/34      DROP ...aceptacion_comercios_num_vinc_old_uso   finalizado   04:41:01 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 35/35    CREATE ...aceptacion_comercios_num_vinc_old_uso   finalizado   04:41:01 PM     00:11.6 
-------------------------------------------------------------------------------------------------
--------------------

# Uso de vinculados todos aceptación comercios

In [32]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_num_vinc_all_uso PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_num_vinc_all_uso STORED AS PARQUET AS
WITH uso_adqui AS
  (SELECT periodo,
          count(*) AS num_vinc_all_uso_adqu
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'adqui'
   AND tipo_cliente = 'todos'
   GROUP BY 1),
     uso_wompi AS
  (SELECT periodo,
          count(*) AS num_vinc_all_uso_womp
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'wompi'
   AND tipo_cliente = 'todos'
   GROUP BY 1)
SELECT a.periodo as fecha_ym,
       a.num_vinc_all_uso_adqu + nvl(b.num_vinc_all_uso_womp, 0) AS num_vinc_all_uso_cumsum_ym
FROM uso_adqui AS a
LEFT JOIN uso_wompi AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_num_vinc_all_uso;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 37/37      DROP ...aceptacion_comercios_num_vinc_all_uso   finalizado   04:41:14 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 38/38    CREATE ...aceptacion_comercios_num_vinc_all_uso   finalizado   04:41:14 PM     00:10.4 
-------------------------------------------------------------------------------------------------
--------------------

# Tabla resultado

In [33]:
sql = """
WITH outcome1 AS
  (SELECT a.fecha_ym,
          a.num_vinc_new,
          a.num_vinc_cumsum,
          a.num_vinc_new_cumsum_ym,
          nvl(b.num_vinc_new_uso_cumsum_ym, 0) AS num_vinc_new_uso_cumsum_ym,
          round(nvl(b.num_vinc_new_uso_cumsum_ym, 0)/a.num_vinc_new_cumsum_ym, 4) AS num_vinc_new_prop_uso,
          nvl(c.num_vinc_old_uso_cumsum_ym, 0) AS num_vinc_old_uso_cumsum_ym,
          nvl(d.num_vinc_all_uso_cumsum_ym, 0) AS num_vinc_all_uso_cumsum_ym,
          round(nvl(d.num_vinc_all_uso_cumsum_ym, 0)/a.num_vinc_cumsum, 4) AS num_vinc_all_prop_uso,
          left(cast(a.fecha_ym AS string), 4) AS YEAR,
          right(cast(a.fecha_ym AS string), 2) AS mes
   FROM proceso.mdo_aceptacion_comercios_vinc AS a
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_new_uso AS b ON a.fecha_ym = b.fecha_ym
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_old_uso AS c ON a.fecha_ym = c.fecha_ym
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_all_uso AS d ON a.fecha_ym = d.fecha_ym),
     outcome2 AS
  (SELECT fecha_ym,
          num_vinc_cumsum AS num_vinc_old,
          cast(cast(YEAR AS int) + 1 AS string) AS YEAR
   FROM outcome1
   WHERE mes = '12')
SELECT a.fecha_ym,
       CONCAT(a.YEAR, '/', a.mes, '/', '01') AS fecha_ym2,
       a.num_vinc_new,
       a.num_vinc_new_cumsum_ym,
       a.num_vinc_new_uso_cumsum_ym,
       a.num_vinc_new_prop_uso,
       b.num_vinc_old,
       a.num_vinc_old_uso_cumsum_ym,
       round(a.num_vinc_old_uso_cumsum_ym/b.num_vinc_old, 4) AS num_vinc_old_prop_uso,
       a.num_vinc_cumsum,
       a.num_vinc_all_uso_cumsum_ym,
       a.num_vinc_all_prop_uso
FROM outcome1 AS a
LEFT JOIN outcome2 AS b ON a.year = b.year
WHERE a.fecha_ym BETWEEN 202201 AND 202511
ORDER BY a.fecha_ym DESC;
"""
# print(sql)
df_outcome = helper.obtener_dataframe(sql)
df_outcome

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 40/40 DATAFRAME                                            ejecutando   04:41:26 PM             

2026-01-16 16:41:27 - [INFO] - 47 filas, 12 columnas, 00:00.4 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 40/40 DATAFRAME                                            finalizado   04:41:26 PM     00:00.6 
-------------------------------------------------------------------------------------------------


,fecha_ym,fecha_ym2,num_vinc_new,num_vinc_new_cumsum_ym,num_vinc_new_uso_cumsum_ym,num_vinc_new_prop_uso,num_vinc_old,num_vinc_old_uso_cumsum_ym,num_vinc_old_prop_uso,num_vinc_cumsum,num_vinc_all_uso_cumsum_ym,num_vinc_all_prop_uso
0,202511.0,2025/11/01,14967,98583,38823,0.3938,309628,103274,0.3335,408211,142097,0.3481
1,202510.0,2025/10/01,16147,83616,34180,0.4088,309628,102731,0.3318,393244,136911,0.3482
2,202509.0,2025/09/01,18340,67469,29340,0.4349,309628,102090,0.3297,377097,131430,0.3485
3,202508.0,2025/08/01,10610,49129,22020,0.4482,309628,101310,0.3272,358757,123330,0.3438
4,202507.0,2025/07/01,7872,38519,19882,0.5162,309628,100437,0.3244,348147,120319,0.3456
5,202506.0,2025/06/01,6189,30647,15970,0.5211,309628,99408,0.3211,340275,115378,0.3391
6,202505.0,2025/05/01,6326,24458,12905,0.5276,309628,98241,0.3173,334086,111146,0.3327
7,202504.0,2025/04/01,5101,18132,9492,0.5235,309628,96434,0.3115,327760,105926,0.3232
8,202503.0,2025/03/01,4730,13031,6541,0.5020,309628,94424,0.3050,322659,100965,0.3129
9,202502.0,2025/02/01,4296,8301,3989,0.4805,309628,91308,0.2949,317929,107771,0.3390


In [ ]:
sql = """
WITH outcome1 AS
  (SELECT a.fecha_ym,
          a.num_vinc_new,
          a.num_vinc_cumsum,
          a.num_vinc_new_cumsum_ym,
          nvl(b.num_vinc_new_uso_cumsum_ym, 0) AS num_vinc_new_uso_cumsum_ym,
          round(nvl(b.num_vinc_new_uso_cumsum_ym, 0)/a.num_vinc_new_cumsum_ym, 4) AS num_vinc_new_prop_uso,
          nvl(c.num_vinc_old_uso_cumsum_ym, 0) AS num_vinc_old_uso_cumsum_ym,
          nvl(d.num_vinc_all_uso_cumsum_ym, 0) AS num_vinc_all_uso_cumsum_ym,
          round(nvl(d.num_vinc_all_uso_cumsum_ym, 0)/a.num_vinc_cumsum, 4) AS num_vinc_all_prop_uso,
          left(cast(a.fecha_ym AS string), 4) AS YEAR,
          right(cast(a.fecha_ym AS string), 2) AS mes
   FROM proceso.mdo_aceptacion_comercios_vinc AS a
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_new_uso AS b ON a.fecha_ym = b.fecha_ym
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_old_uso AS c ON a.fecha_ym = c.fecha_ym
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_all_uso AS d ON a.fecha_ym = d.fecha_ym),
     outcome2 AS
  (SELECT fecha_ym,
          num_vinc_cumsum AS num_vinc_old,
          cast(cast(YEAR AS int) + 1 AS string) AS YEAR
   FROM outcome1
   WHERE mes = '12')
SELECT a.fecha_ym,
       CONCAT(a.YEAR, '/', a.mes, '/', '01') AS fecha_ym2,
       a.num_vinc_new,
       a.num_vinc_new_cumsum_ym,
       a.num_vinc_new_uso_cumsum_ym,
       a.num_vinc_new_prop_uso,
       b.num_vinc_old,
       a.num_vinc_old_uso_cumsum_ym,
       round(a.num_vinc_old_uso_cumsum_ym/b.num_vinc_old, 4) AS num_vinc_old_prop_uso,
       a.num_vinc_cumsum,
       a.num_vinc_all_uso_cumsum_ym,
       a.num_vinc_all_prop_uso
FROM outcome1 AS a
LEFT JOIN outcome2 AS b ON a.year = b.year
WHERE a.fecha_ym BETWEEN 202201 AND 202511
ORDER BY a.fecha_ym DESC;
"""
# print(sql)
df_outcome = helper.obtener_dataframe(sql)
df_outcome

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 100/100 DATAFRAME                                           descargando   12:14:47 AM             

2025-12-12 00:14:48 - [INFO] - 47 filas, 12 columnas, 00:00.6 consultando, 00:00.2 descargando, 00:00.0 convirtiendo


 100/100 DATAFRAME                                            finalizado   12:14:47 AM     00:00.9 
---------------------------------------------------------------------------------------------------


,fecha_ym,fecha_ym2,num_vinc_new,num_vinc_new_cumsum_ym,num_vinc_new_uso_cumsum_ym,num_vinc_new_prop_uso,num_vinc_old,num_vinc_old_uso_cumsum_ym,num_vinc_old_prop_uso,num_vinc_cumsum,num_vinc_all_uso_cumsum_ym,num_vinc_all_prop_uso
0,202511.0,2025/11/01,14983,98681,37253,0.3775,309667,96086,0.3103,408348,133339,0.3265
1,202510.0,2025/10/01,16180,83698,32665,0.3903,309667,95557,0.3086,393365,128222,0.3260
2,202509.0,2025/09/01,18355,67518,27900,0.4132,309667,94927,0.3065,377185,122827,0.3256
3,202508.0,2025/08/01,10621,49163,22995,0.4677,309667,94168,0.3041,358830,117163,0.3265
4,202507.0,2025/07/01,7876,38542,18936,0.4913,309667,93317,0.3013,348209,112253,0.3224
5,202506.0,2025/06/01,6191,30666,15268,0.4979,309667,92331,0.2982,340333,107599,0.3162
6,202505.0,2025/05/01,6337,24475,12327,0.5037,309667,91225,0.2946,334142,103552,0.3099
7,202504.0,2025/04/01,5104,18138,9031,0.4979,309667,89479,0.2890,327805,98510,0.3005
8,202503.0,2025/03/01,4731,13034,6196,0.4754,309667,87526,0.2826,322701,93722,0.2904
9,202502.0,2025/02/01,4296,8303,3760,0.4528,309667,84511,0.2729,317970,88271,0.2776


In [ ]:
df_outcome.to_excel('main_data/evolucion_vinculacion_y_uso_adquirencia_y_wompi.xlsx')

# Eliminación tablas proceso.

In [3]:
# Eliminación tablas proceso.
tablas_borrar = ['proceso.mdo_aceptacion_comercios_hist_vinc_adqu', 'proceso.mdo_aceptacion_comercios_hist_vinc_wompi', 'proceso.mdo_aceptacion_comercios_vinc', 'proceso.mdo_aceptacion_comercios_num_vinc_new_uso', 'proceso.mdo_aceptacion_comercios_num_vinc_old_uso', 'proceso.mdo_aceptacion_comercios_num_vinc_all_uso']
for tabla in tablas_borrar:
    sql_drop = f"""DROP TABLE IF EXISTS {tabla} PURGE;"""
    helper.ejecutar_consulta(sql_drop)

2026-02-05 14:19:30 - [INFO] - Transcurrido: 1770319170, Tiempo de Refresco = 1000


------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 1/1 DROP ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   02:19:30 PM     00:00.1 
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 2/2 DROP ..._aceptacion_comercios_hist_vinc_wompi   finalizado   02:19:30 PM     00:00.1 
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------